In [ ]:
import json
import os

import requests

In [ ]:
os.environ["BALANCE_API_BASE_URL"] = (
    "https://microsoft-foundry-playground-balance-mock-api.9uxguk.easypanel.host"
)

In [ ]:
os.environ.get("BALANCE_API_BASE_URL", "http://localhost:8000")

In [ ]:
def get_base_url() -> str:
    return os.environ.get("BALANCE_API_BASE_URL", "http://localhost:8000")


def call_balance_api(customer_id: str) -> dict:
    url = f"{get_base_url().rstrip('/')}/api/balance"
    payload = {"customer_id": customer_id}
    response = requests.post(url, json=payload, timeout=5)
    try:
        data = response.json()
    except json.JSONDecodeError:
        data = {"raw_text": response.text}
    return {
        "status_code": response.status_code,
        "data": data,
    }

In [ ]:
result = call_balance_api("customer-002")
print(json.dumps(result, indent=2, ensure_ascii=False))

---
## Tokens Test

In [2]:
import os

import dotenv
import requests

In [3]:
dotenv.load_dotenv()

True

In [ ]:
KEYCLOAK_BASE_URL = os.getenv("KEYCLOACK_BASE_URL")
REALM_NAME = os.getenv("REALM_NAME")
CLIENT_ID = os.getenv("CLIENT_ID")
CLIENT_SECRET = os.getenv("CLIENT_SECRET")
USER_1_NAME = os.getenv("USER_1_NAME")
USER_2_NAME = os.getenv("USER_2_NAME")
USER_1_PASSWORD = os.getenv("USER_1_PASSWORD")
USER_2_PASSWORD = os.getenv("USER_2_PASSWORD")

TOKEN_URL = f"{KEYCLOAK_BASE_URL}/realms/{REALM_NAME}/protocol/openid-connect/token"

In [ ]:
def get_access_token(username: str, password: str):
    data = {
        "grant_type": "password",
        "client_id": CLIENT_ID,
        "client_secret": CLIENT_SECRET,
        "username": username,
        "password": password,
    }

    headers = {"Content-Type": "application/x-www-form-urlencoded"}

    response = requests.post(TOKEN_URL, data=data, headers=headers)

    if response.status_code != 200:
        raise Exception(f"Error {response.status_code}: {response.text}")

    return response.json()["access_token"]

In [ ]:
token = get_access_token(USER_1_NAME, USER_1_PASSWORD)
print(token)

In [ ]:
token = get_access_token(USER_2_NAME, USER_2_PASSWORD)
print(token)

### Use Token to Call API

In [ ]:
response = requests.post(
    "http://localhost:8000/api/balance",
    headers={
        "Authorization": f"Bearer {token}",
        "Content-Type": "application/json",
    },
    json={},
    timeout=10,
)

print(response.status_code)
print(response.text)

In [ ]:
response = requests.post("http://localhost:8000/api/balance", json={})

assert str(response) == '<Response [401]>'

### Chatbot API

In [20]:
response = requests.post("http://localhost:5000/chat", json={"message": "hola"})

In [21]:
print(response.status_code)
print(response.text)

200
{"response":{"type":"text","text":{"value":"Hola, ¿en qué puedo ayudarte hoy? ¿Quieres consultar el saldo de tu cuenta? Por favor, dime tu identificador de cliente.","annotations":[]}}}
